In [5]:
import pandas as pd
import numpy as np
import re
import json
import time
from tqdm import tqdm
from nltk.stem.snowball import RussianStemmer
import bm25s

tqdm.pandas()
pd.set_option('display.max_columns', 100)

In [6]:
train = pd.read_parquet('../dataset/train.parquet')
benchmark_items = pd.read_parquet('../dataset/benchmark_items.parquet')
benchmark_queries = pd.read_parquet('../dataset/benchmark_queries.parquet')

# id — всегда строки
train['item_id'] = train['item_id'].astype(str)
benchmark_items['item_id'] = benchmark_items['item_id'].astype(str)
benchmark_queries['query_id'] = benchmark_queries['query_id'].astype(str)

print("train:", train.shape)
print("benchmark_items:", benchmark_items.shape)
print("benchmark_queries:", benchmark_queries.shape)

train: (497673, 19)
benchmark_items: (189212, 14)
benchmark_queries: (2452, 6)


In [12]:
def to_float(s):
    if pd.isna(s):
        return np.nan
    s = str(s).replace(',', '.').replace(' ', '')
    try:
        return float(s)
    except Exception:
        return np.nan

for df in [train, benchmark_items]:
    df['item_price'] = df['item_price'].apply(to_float)
    df['item_longitude'] = df['item_longitude'].apply(to_float)
    df['item_latitude'] = df['item_latitude'].apply(to_float)
    df['item_rating'] = pd.to_numeric(df['item_rating'], errors='coerce')
    df['item_rating_reviews_count'] = pd.to_numeric(df['item_rating_reviews_count'], errors='coerce')

for df in [train, benchmark_items]:
    df['item_rating'] = df['item_rating'].fillna(-1)
    df['item_rating_reviews_count'] = df['item_rating_reviews_count'].fillna(-1)

print("Числовые поля приведены.")
print(benchmark_items[['item_price', 'item_rating', 'item_rating_reviews_count']].describe())

Числовые поля приведены.
         item_price    item_rating  item_rating_reviews_count
count  1.892120e+05  189212.000000              189212.000000
mean   1.986029e+07       4.249478                  62.334535
std    3.730628e+09       1.822869                 224.202924
min   -1.000000e+00      -1.000000                  -1.000000
25%    3.800000e+02       4.783784                   4.000000
50%    1.000000e+03       4.997001                  15.000000
75%    2.500000e+03       5.000000                  49.000000
max    7.777778e+11       5.000000               18291.000000


In [ ]:
def make_query_key(df):
    cols = ['search_query', 'search_location_id', 'search_is_delivery_search',
            'search_infm_params_text', 'search_category']
    return df[cols].fillna('__NA__').astype(str).agg('||'.join, axis=1)

train['query_key'] = make_query_key(train)

before = len(train)
train = train.drop_duplicates(subset=['query_key', 'item_id']).reset_index(drop=True)
print(f"train: {before} -> {len(train)} после удаления дублей query_key+item_id")

In [ ]:
TOKEN_RE = re.compile(r'[а-яa-z0-9]+')
_STEMMER = RussianStemmer()
_STEM_CACHE = {}

def tokenize_stem(text):
    if not isinstance(text, str) or not text:
        return []
    text = text.lower().replace('ё', 'е')
    tokens = TOKEN_RE.findall(text)
    out = []
    cache = _STEM_CACHE
    stemmer = _STEMMER
    for t in tokens:
        if len(t) <= 1:
            continue
        s = cache.get(t)
        if s is None:
            s = stemmer.stem(t)
            cache[t] = s
        out.append(s)
    return out

print(tokenize_stem("Монтаж видеодомофонов в Москве, недорого!"))

In [ ]:
titles = benchmark_items['item_title_raw'].fillna('').astype(str)
descs = benchmark_items['item_description_raw'].fillna('').astype(str)
params = benchmark_items['item_infm_params_text'].fillna('').astype(str)

benchmark_items['item_text'] = (
    titles + ' ' + titles + ' ' + titles + ' ' + descs + ' ' + params
)

print("Пример текста объявления:")
print(benchmark_items['item_text'].iloc[0][:300])

In [ ]:
start = time.time()
item_tokens = []
for t in tqdm(benchmark_items['item_text'].tolist(), desc="Токенизация benchmark_items"):
    item_tokens.append(tokenize_stem(t))
benchmark_items['tokens'] = item_tokens
elapsed = time.time() - start
print(f"Токенизация заняла {elapsed:.1f} сек. ({elapsed/60:.1f} мин.)")
print(f"Уникальных токенов в кэше стеммера: {len(_STEM_CACHE):,}")
print("Пример токенов:", benchmark_items['tokens'].iloc[0][:20])

In [ ]:
with open('../dataset/benchmark_items_tokens.jsonl', 'w', encoding='utf-8') as f:
    for item_id, toks in zip(benchmark_items['item_id'], benchmark_items['tokens']):
        f.write(json.dumps({'item_id': item_id, 'tokens': toks}, ensure_ascii=False) + '\n')

print("Токены сохранены в dataset/benchmark_items_tokens.jsonl")

In [ ]:
start = time.time()
corpus_tokens = benchmark_items['tokens'].tolist()

retriever = bm25s.BM25()
retriever.index(corpus_tokens)
print(f"BM25 индекс построен за {time.time() - start:.1f} сек.")

In [ ]:
retriever.save('../dataset/bm25_index')
print("BM25 индекс сохранён в dataset/bm25_index")

In [ ]:
bench_item_ids = set(benchmark_items['item_id'])
valid_mask = train['item_id'].isin(bench_item_ids)
valid_pairs = train[valid_mask].copy()

print(f"Пар train с item_id из benchmark_items: {len(valid_pairs)}")

valid_groups = valid_pairs.groupby('query_key').agg(
    search_query=('search_query', 'first'),
    search_infm=('search_infm_params_text', 'first'),
    search_cat=('search_category', 'first'),
    search_location=('search_location_id', 'first'),
    relevant_items=('item_id', lambda x: list(set(x))),
).reset_index()

print(f"Валидационных query_key: {len(valid_groups)}")
print("Пример:")
display(valid_groups.head(3))

In [ ]:
def build_query_text(row):
    parts = [str(row['search_query'])]
    if not pd.isna(row['search_infm']):
        parts.append(str(row['search_infm']))
    parts.append(str(row['search_cat']))
    return ' '.join(parts)

valid_groups['query_text_full'] = valid_groups.apply(build_query_text, axis=1)

query_texts = valid_groups['query_text_full'].tolist()
query_tokens_list = [tokenize_stem(t) for t in tqdm(query_texts, desc="Токенизация запросов")]
valid_groups['query_tokens'] = query_tokens_list

print("Пример запроса:", valid_groups['query_text_full'].iloc[0])
print("Пример токенов:", valid_groups['query_tokens'].iloc[0])

In [ ]:
item_ids_array = benchmark_items['item_id'].values

K = 50
BATCH_SIZE = 256

query_tokens_list = valid_groups['query_tokens'].tolist()
n_queries = len(query_tokens_list)
print(f"Всего запросов для оценки: {n_queries}")

topk_ids_per_query = [None] * n_queries

start = time.time()
for start_i in tqdm(range(0, n_queries, BATCH_SIZE), desc="BM25 batch retrieve"):
    end_i = min(start_i + BATCH_SIZE, n_queries)
    batch_queries = query_tokens_list[start_i:end_i]

    safe_batch = [q if q else ['_empty_'] for q in batch_queries]

    results, scores = retriever.retrieve(safe_batch, k=K, show_progress=False)

    for j, q in enumerate(batch_queries):
        if not q:
            topk_ids_per_query[start_i + j] = set()
        else:
            topk_ids_per_query[start_i + j] = set(item_ids_array[results[j]])

elapsed = time.time() - start
print(f"\nБатчевый retrieve занял {elapsed:.1f} сек. ({elapsed/60:.1f} мин.)")

In [ ]:
recalls = []
for pos, (_, row) in enumerate(valid_groups.iterrows()):
    relevant = set(row['relevant_items'])
    retrieved = topk_ids_per_query[pos]
    if not relevant:
        recalls.append(0.0)
        continue
    hit = len(retrieved & relevant)
    recalls.append(hit / len(relevant))

recalls = np.array(recalls)
valid_groups['recall'] = recalls

print("Baseline BM25 Recall@50:")
print(f"  mean    = {recalls.mean():.4f}")
print(f"  median  = {np.median(recalls):.4f}")
print(f"  Recall=0: {(recalls == 0).mean():.4f}")
print(f"  Recall=1: {(recalls == 1).mean():.4f}")
print(f"  Всего запросов: {len(recalls)}")

In [ ]:
failed = valid_groups[valid_groups['recall'] == 0].copy()
print(f"Запросов с Recall=0: {len(failed)} из {len(valid_groups)}")

for _, row in failed.head(10).iterrows():
    print(f"\nquery: {row['search_query']}")
    print(f"  infm: {row['search_infm']}")
    print(f"  n_relevant: {len(row['relevant_items'])}")
    print(f"  relevant example: {row['relevant_items'][:2]}")

    q_tokens = row['query_tokens']
    if q_tokens:
        res, _ = retriever.retrieve([q_tokens], k=5)
        top5 = item_ids_array[res[0]]
        print(f"  top-5 BM25: {list(top5)}")

In [ ]:
perfect = valid_groups[valid_groups['recall'] == 1.0]
print(f"Запросов с Recall=1: {len(perfect)} из {len(valid_groups)}")

for _, row in perfect.head(10).iterrows():
    print(f"query: {row['search_query']} | n_relevant={len(row['relevant_items'])}")

In [ ]:
import numpy as np
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import time

In [ ]:
import bm25s
import time

titles_tokens = []
descs_tokens = []
params_tokens = []

for t, d, p in tqdm(
    zip(benchmark_items['item_title_raw'],
        benchmark_items['item_description_raw'],
        benchmark_items['item_infm_params_text']),
    total=len(benchmark_items),
    desc="Токенизация трёх полей"
):
    titles_tokens.append(tokenize_stem('' if pd.isna(t) else str(t)))
    descs_tokens.append(tokenize_stem('' if pd.isna(d) else str(d))[:100])  # ограничим 100 токенов
    params_tokens.append(tokenize_stem('' if pd.isna(p) else str(p))[:100])

benchmark_items['tokens_title'] = titles_tokens
benchmark_items['tokens_desc'] = descs_tokens
benchmark_items['tokens_params'] = params_tokens

print("Строим BM25 title...")
r_title = bm25s.BM25()
r_title.index(titles_tokens)

print("Строим BM25 desc...")
r_desc = bm25s.BM25()
r_desc.index(descs_tokens)

print("Строим BM25 params...")
r_params = bm25s.BM25()
r_params.index(params_tokens)

print("Строим BM25 full...")
r_full = bm25s.BM25()
r_full.index(benchmark_items['tokens'].tolist())

with open('../dataset/item_tokens_fields.jsonl', 'w', encoding='utf-8') as f:
    for iid, tt, dt, pt in zip(benchmark_items['item_id'], titles_tokens, descs_tokens, params_tokens):
        f.write(json.dumps({'item_id': iid, 'title': tt, 'desc': dt, 'params': pt}, ensure_ascii=False) + '\n')

print("Готово.")

In [ ]:
r_full.save('../dataset/bm25_index')
print("Сохранён r_full в ../dataset/bm25_index")

In [ ]:
r_title.save('../dataset/bm25_index_title')
r_desc.save('../dataset/bm25_index_desc')
r_params.save('../dataset/bm25_index_params')
print("Сохранены индексы: title, desc, params")

In [ ]:
K_BM25 = 500

def batch_bm25(retriever, query_tokens_list, k, name, batch_size=128):
    all_ids = [None] * len(query_tokens_list)
    all_scores = [None] * len(query_tokens_list)
    for start_i in tqdm(range(0, len(query_tokens_list), batch_size), desc=f"BM25 {name} top-{k}"):
        end_i = min(start_i + batch_size, len(query_tokens_list))
        batch_q = query_tokens_list[start_i:end_i]
        safe_batch = [q if q else ['_empty_'] for q in batch_q]
        res, sc = retriever.retrieve(safe_batch, k=k, show_progress=False)
        for j in range(len(batch_q)):
            all_ids[start_i + j] = res[j]
            all_scores[start_i + j] = sc[j]
    return all_ids, all_scores

start = time.time()
bm25_full_ids, bm25_full_scores = batch_bm25(r_full, query_tokens_list, K_BM25, "full")
print(f"BM25 full занял {time.time() - start:.1f} сек.")

start = time.time()
bm25_title_ids, bm25_title_scores = batch_bm25(r_title, query_tokens_list, K_BM25, "title")
print(f"BM25 title занял {time.time() - start:.1f} сек.")

start = time.time()
bm25_desc_ids, bm25_desc_scores = batch_bm25(r_desc, query_tokens_list, K_BM25, "desc")
print(f"BM25 desc занял {time.time() - start:.1f} сек.")

start = time.time()
bm25_params_ids, bm25_params_scores = batch_bm25(r_params, query_tokens_list, K_BM25, "params")
print(f"BM25 params занял {time.time() - start:.1f} сек.")

In [ ]:
import random
random.seed(42)

MAX_POS_PER_QUERY = 10
N_NEG_PER_QUERY = 50

item_to_pos = {iid: i for i, iid in enumerate(benchmark_items['item_id'])}
item_tokens_title_list = benchmark_items['tokens_title'].tolist()
item_tokens_desc_list = benchmark_items['tokens_desc'].tolist()
item_tokens_params_list = benchmark_items['tokens_params'].tolist()

def char_3grams(text):
    text = str(text).lower()
    text = re.sub(r'[^а-яa-z0-9]', '', text)
    if len(text) < 3:
        return set()
    return {text[i:i+3] for i in range(len(text) - 2)}

query_chargrams = []
for pos in range(len(valid_groups)):
    q_text = valid_groups.iloc[pos]['search_query']
    query_chargrams.append(char_3grams(q_text))

item_chargrams = []
for t in benchmark_items['item_title_raw']:
    item_chargrams.append(char_3grams(t))

pairs = []
for pos in tqdm(range(len(valid_groups)), desc="Building pairs"):
    relevant = set(valid_groups.iloc[pos]['relevant_items'])

    pool = set()
    for arr in [bm25_full_ids[pos], bm25_title_ids[pos], bm25_desc_ids[pos], bm25_params_ids[pos]]:
        pool.update(arr.tolist())

    pos_items = [item_to_pos[i] for i in relevant if i in item_to_pos]
    random.shuffle(pos_items)
    pos_items = pos_items[:MAX_POS_PER_QUERY]
    for ip in pos_items:
        pairs.append((pos, ip, 1))

    neg_pool = list(pool - set(pos_items))
    random.shuffle(neg_pool)
    for ip in neg_pool[:N_NEG_PER_QUERY]:
        pairs.append((pos, ip, 0))

pairs_df = pd.DataFrame(pairs, columns=['query_pos', 'item_pos', 'label'])
print(f"Всего пар: {len(pairs_df)}, позитивов: {pairs_df['label'].sum()}")

In [ ]:
bi = benchmark_items.reset_index(drop=True)
item_microcat_arr = bi['item_microcat_id'].values
item_loc_arr = bi['item_location_id'].values
item_rating_arr = bi['item_rating'].values
item_reviews_arr = bi['item_rating_reviews_count'].values
item_price_arr = bi['item_price'].fillna(0).clip(lower=0).values
item_price_log_arr = np.log1p(item_price_arr)
item_title_len_arr = bi['item_title_raw'].fillna('').astype(str).str.len().values

item_ids_array = bi['item_id'].values

def score_rank_for(ids_arr, scores_arr, qp, ip):
    ids = ids_arr[qp]
    where = np.where(ids == ip)[0]
    if len(where) == 0:
        return 9999, 0.0
    rank = int(where[0])
    return rank, float(scores_arr[qp][rank])

print("Считаем признаки (может занять 10-15 минут)...")
features = {
    'bm25_full_score': [],
    'bm25_full_rank': [],
    'bm25_title_score': [],
    'bm25_title_rank': [],
    'bm25_desc_score': [],
    'bm25_desc_rank': [],
    'bm25_params_score': [],
    'bm25_params_rank': [],
    'overlap_recall': [],
    'overlap_jaccard': [],
    'title_overlap_recall': [],
    'desc_overlap_recall': [],
    'params_overlap_recall': [],
    'char3_jaccard': [],
    'loc_match': [],
    'cat_match': [],
    'microcat_match': [],
    'item_rating': [],
    'item_reviews': [],
    'item_price_log': [],
    'item_title_len': [],
}

q_tokens_list = valid_groups['query_tokens'].tolist()
q_loc_list = valid_groups['search_location'].astype(str).tolist()
q_cat_list = valid_groups['search_cat'].astype(str).tolist()

for qi, ip, lbl in tqdm(zip(pairs_df['query_pos'], pairs_df['item_pos'], pairs_df['label']),
                        total=len(pairs_df), desc="Features"):
    qt = q_tokens_list[qi]
    qs = set(qt)

    tt = item_tokens_title_list[ip]
    dt = item_tokens_desc_list[ip]
    pt = item_tokens_params_list[ip]
    all_tokens = tt + dt + pt
    at = set(all_tokens)

    rf, sf = score_rank_for(bm25_full_ids, bm25_full_scores, qi, ip)
    rt, st = score_rank_for(bm25_title_ids, bm25_title_scores, qi, ip)
    rd, sd = score_rank_for(bm25_desc_ids, bm25_desc_scores, qi, ip)
    rp, sp = score_rank_for(bm25_params_ids, bm25_params_scores, qi, ip)

    features['bm25_full_score'].append(sf)
    features['bm25_full_rank'].append(rf)
    features['bm25_title_score'].append(st)
    features['bm25_title_rank'].append(rt)
    features['bm25_desc_score'].append(sd)
    features['bm25_desc_rank'].append(rd)
    features['bm25_params_score'].append(sp)
    features['bm25_params_rank'].append(rp)

    if qs and at:
        inter = len(qs & at)
        features['overlap_recall'].append(inter / len(qs))
        features['overlap_jaccard'].append(inter / len(qs | at))
    else:
        features['overlap_recall'].append(0.0)
        features['overlap_jaccard'].append(0.0)

    ts = set(tt)
    ds = set(dt)
    ps = set(pt)
    features['title_overlap_recall'].append(len(qs & ts) / len(qs) if qs else 0.0)
    features['desc_overlap_recall'].append(len(qs & ds) / len(qs) if qs else 0.0)
    features['params_overlap_recall'].append(len(qs & ps) / len(qs) if qs else 0.0)

    qcg = query_chargrams[qi]
    icg = item_chargrams[ip]
    if qcg and icg:
        features['char3_jaccard'].append(len(qcg & icg) / len(qcg | icg))
    else:
        features['char3_jaccard'].append(0.0)

    features['loc_match'].append(int(q_loc_list[qi] == str(item_loc_arr[ip])))
    features['cat_match'].append(int(q_cat_list[qi] == str(item_microcat_arr[ip])))
    features['microcat_match'].append(int(q_cat_list[qi] == str(item_microcat_arr[ip])))

    features['item_rating'].append(float(item_rating_arr[ip]))
    features['item_reviews'].append(float(item_reviews_arr[ip]))
    features['item_price_log'].append(float(item_price_log_arr[ip]))
    features['item_title_len'].append(int(item_title_len_arr[ip]))

for k, v in features.items():
    pairs_df[k] = v

pairs_df.to_parquet('../dataset/train_pairs_features.parquet', index=False)
print("Признаки сохранены в dataset/train_pairs_features.parquet")
print(f"Итого: {len(pairs_df)} пар, {pairs_df.shape[1]} колонок")

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
import numpy as np

pairs_df = pd.read_parquet('../dataset/train_pairs_features.parquet')
print(f"Пар: {len(pairs_df)}, позитивов: {pairs_df['label'].sum()}")

feature_cols = [
    'bm25_full_score', 'bm25_full_rank',
    'bm25_title_score', 'bm25_title_rank',
    'bm25_desc_score', 'bm25_desc_rank',
    'bm25_params_score', 'bm25_params_rank',
    'overlap_recall', 'overlap_jaccard',
    'title_overlap_recall', 'desc_overlap_recall', 'params_overlap_recall',
    'char3_jaccard',
    'loc_match', 'cat_match', 'microcat_match',
    'item_rating', 'item_reviews', 'item_price_log', 'item_title_len',
]

query_text_by_pos = valid_groups['search_query'].to_dict()
pairs_df['query_text_g'] = pairs_df['query_pos'].map(query_text_by_pos)
print(f"Уникальных групп: {pairs_df['query_text_g'].nunique()}")

In [ ]:
N_FOLDS = 5
gkf = GroupKFold(n_splits=N_FOLDS)

fold_metrics = []
models = []

lgb_params = dict(
    objective='binary',
    metric='auc',
    learning_rate=0.03,
    num_leaves=63,
    min_child_samples=30,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    lambda_l1=0.1,
    lambda_l2=0.1,
    n_estimators=2000,
    n_jobs=8,
    random_state=42,
    verbose=-1,
)

def compute_recall_at_k_for_fold(model, valid_df, pairs_df, valid_groups,
                                  bm25_full_ids, item_ids_array, K=50):
    valid_q_positions = valid_df['query_pos'].unique()

    recalls = []
    for qp in valid_q_positions:
        candidates = pairs_df[pairs_df['query_pos'] == qp]
        if len(candidates) == 0:
            recalls.append(0.0)
            continue

        X_q = candidates[feature_cols].values
        scores = model.predict_proba(X_q)[:, 1]
        top_idx = np.argsort(-scores)[:K]

        top_item_pos = candidates.iloc[top_idx]['item_pos'].values
        top_item_ids = set(item_ids_array[top_item_pos])

        relevant = set(valid_groups.iloc[qp]['relevant_items'])
        if not relevant:
            recalls.append(0.0)
        else:
            recalls.append(len(top_item_ids & relevant) / len(relevant))

    return np.array(recalls)


for fold, (train_idx, valid_idx) in enumerate(gkf.split(pairs_df, groups=pairs_df['query_text_g'])):
    print(f"\n{'='*60}")
    print(f"FOLD {fold + 1}/{N_FOLDS}")
    print(f"{'='*60}")

    train_df = pairs_df.iloc[train_idx]
    valid_df = pairs_df.iloc[valid_idx]

    X_train = train_df[feature_cols].values
    y_train = train_df['label'].values
    X_valid = valid_df[feature_cols].values
    y_valid = valid_df['label'].values

    scale = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
    print(f"Train: {len(train_df)}, Valid: {len(valid_df)}, scale_pos_weight: {scale:.2f}")

    model = lgb.LGBMClassifier(**lgb_params, scale_pos_weight=scale)
    model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric='auc',
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)],
    )

    preds = model.predict_proba(X_valid)[:, 1]
    auc = roc_auc_score(y_valid, preds)

    valid_q_positions = valid_df['query_pos'].unique()
    recalls = compute_recall_at_k_for_fold(
        model, valid_df, pairs_df, valid_groups,
        bm25_full_ids, item_ids_array, K=50,
    )
    recall_for_all = np.zeros(len(valid_groups))
    recall_for_all[valid_q_positions] = recalls
    mask = np.zeros(len(valid_groups), dtype=bool)
    mask[valid_q_positions] = True

    fold_mean = recalls.mean()
    print(f"Fold {fold+1}: AUC={auc:.4f}, Recall@50 (на валид-запросах фолда)={fold_mean:.4f}")

    fold_metrics.append({
        'fold': fold + 1,
        'auc': auc,
        'recall_at_50': fold_mean,
        'best_iter': model.best_iteration_,
        'n_valid_queries': len(valid_q_positions),
    })
    models.append(model)

metrics_df = pd.DataFrame(fold_metrics)
print("\n" + "=" * 60)
print("ИТОГИ K-FOLD:")
print(metrics_df.to_string(index=False))
print("=" * 60)
print(f"Средний Recall@50: {metrics_df['recall_at_50'].mean():.4f}")
print(f"Средний AUC:      {metrics_df['auc'].mean():.4f}")

In [ ]:
importance_all = np.zeros(len(feature_cols))
for m in models:
    importance_all += m.feature_importances_
importance_all /= len(models)

importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': importance_all,
}).sort_values('importance', ascending=False)

print("Усреднённая важность признаков:")
print(importance_df.to_string(index=False))

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'][::-1], importance_df['importance'][::-1])
plt.xlabel('Importance')
plt.title('Feature importance (avg over 5 folds)')
plt.tight_layout()
plt.show()

In [ ]:
best_iter_avg = int(np.mean([m.best_iteration_ for m in models]))
print(f"Средний best_iteration: {best_iter_avg}")

X_all = pairs_df[feature_cols].values
y_all = pairs_df['label'].values

scale = (y_all == 0).sum() / max((y_all == 1).sum(), 1)

final_params = dict(lgb_params)
final_params.pop('n_estimators', None)

final_model = lgb.LGBMClassifier(
    **final_params,
    n_estimators=best_iter_avg,
    scale_pos_weight=scale,
)
final_model.fit(X_all, y_all)
print(f"Финальная модель обучена на {len(pairs_df)} парах.")

import pickle
with open('../dataset/lgbm_reranker.pkl', 'wb') as f:
    pickle.dump({'model': final_model, 'feature_cols': feature_cols}, f)
print("Модель сохранена в dataset/lgbm_reranker.pkl")

In [ ]:
recalls_final = []
for qp in tqdm(range(len(valid_groups)), desc="Final Recall@50"):
    candidates = pairs_df[pairs_df['query_pos'] == qp]
    if len(candidates) == 0:
        recalls_final.append(0.0)
        continue

    X_q = candidates[feature_cols].values
    scores = final_model.predict_proba(X_q)[:, 1]
    top_idx = np.argsort(-scores)[:50]

    top_item_pos = candidates.iloc[top_idx]['item_pos'].values
    top_item_ids = set(item_ids_array[top_item_pos])

    relevant = set(valid_groups.iloc[qp]['relevant_items'])
    recalls_final.append(len(top_item_ids & relevant) / len(relevant) if relevant else 0.0)

recalls_final = np.array(recalls_final)
print(f"\nФинальный Recall@50 (train queries):")
print(f"  mean    = {recalls_final.mean():.4f}")
print(f"  median  = {np.median(recalls_final):.4f}")
print(f"  Recall=0: {(recalls_final == 0).mean():.4f}")
print(f"  Recall=1: {(recalls_final == 1).mean():.4f}")
print(f"\nBM25 top-50 baseline: 0.3165")
print(f"BM25 top-200 ceiling: 0.5825")

In [ ]:
cnt = pairs_df.groupby('query_pos').size()
print("Кандидатов в pairs_df на запрос:")
print(cnt.describe())

pos_cnt = pairs_df[pairs_df['label'] == 1].groupby('query_pos').size()
print("\nПозитивов в pairs_df на запрос:")
print(pos_cnt.describe())

rel_cnt = valid_groups['relevant_items'].apply(len)
print("\nРеальных позитивов на запрос (из valid_groups):")
print(rel_cnt.describe())

bm25_cnt = [len(x) for x in bm25_full_ids]
print(f"\nBM25 top-500 размер: min={min(bm25_cnt)}, max={max(bm25_cnt)}, mean={np.mean(bm25_cnt):.1f}")

In [ ]:
random.seed(42)
N_EVAL_QUERIES = 5000
eval_q_positions = random.sample(range(len(valid_groups)), min(N_EVAL_QUERIES, len(valid_groups)))
print(f"Оцениваем на {len(eval_q_positions)} запросах")

def compute_features_for_query(qp):
    qt = q_tokens_list[qp]
    qs = set(qt)
    q_loc = q_loc_list[qp]
    q_cat = q_cat_list[qp]
    qcg = query_chargrams[qp]

    rows = []
    item_pos_list = []
    for rank, ip in enumerate(bm25_full_ids[qp]):
        ip = int(ip)
        item_pos_list.append(ip)

        tt = item_tokens_title_list[ip]
        dt = item_tokens_desc_list[ip]
        pt = item_tokens_params_list[ip]
        at = set(tt + dt + pt)
        ts = set(tt); ds = set(dt); ps = set(pt)

        rf, sf = score_rank_for(bm25_full_ids, bm25_full_scores, qp, ip)
        rt, st = score_rank_for(bm25_title_ids, bm25_title_scores, qp, ip)
        rd, sd = score_rank_for(bm25_desc_ids, bm25_desc_scores, qp, ip)
        rp, sp = score_rank_for(bm25_params_ids, bm25_params_scores, qp, ip)

        if qs and at:
            inter = len(qs & at)
            ovr = inter / len(qs)
            ovj = inter / len(qs | at)
        else:
            ovr = ovj = 0.0
        tor = len(qs & ts) / len(qs) if qs else 0.0
        dor = len(qs & ds) / len(qs) if qs else 0.0
        por = len(qs & ps) / len(qs) if qs else 0.0

        icg = item_chargrams[ip]
        if qcg and icg:
            c3 = len(qcg & icg) / len(qcg | icg)
        else:
            c3 = 0.0

        rows.append([
            sf, rf, st, rt, sd, rd, sp, rp,
            ovr, ovj, tor, dor, por, c3,
            int(q_loc == str(item_loc_arr[ip])),
            int(q_cat == str(item_microcat_arr[ip])),
            int(q_cat == str(item_microcat_arr[ip])),
            float(item_rating_arr[ip]),
            float(item_reviews_arr[ip]),
            float(item_price_log_arr[ip]),
            int(item_title_len_arr[ip]),
        ])
    return item_pos_list, np.array(rows, dtype=np.float32)

In [ ]:
recalls_honest = []
K_FINAL = 50

start = time.time()
for qp in tqdm(eval_q_positions, desc="Честный Recall@50"):
    item_pos_list, X_q = compute_features_for_query(qp)
    scores = final_model.predict_proba(X_q)[:, 1]
    top_idx = np.argsort(-scores)[:K_FINAL]
    top_item_ids = set(item_ids_array[[item_pos_list[i] for i in top_idx]])

    relevant = set(valid_groups.iloc[qp]['relevant_items'])
    recalls_honest.append(len(top_item_ids & relevant) / len(relevant) if relevant else 0.0)

recalls_honest = np.array(recalls_honest)
print(f"\nЧЕСТНЫЙ Recall@50 на {len(eval_q_positions)} запросах:")
print(f"  mean    = {recalls_honest.mean():.4f}")
print(f"  median  = {np.median(recalls_honest):.4f}")
print(f"  Recall=0: {(recalls_honest == 0).mean():.4f}")
print(f"  Recall=1: {(recalls_honest == 1).mean():.4f}")
print(f"Время: {time.time() - start:.1f} сек.")

In [ ]:
ceiling_500 = []
for qp in eval_q_positions:
    bm25_ids = set(item_ids_array[bm25_full_ids[qp]])
    relevant = set(valid_groups.iloc[qp]['relevant_items'])
    ceiling_500.append(len(bm25_ids & relevant) / len(relevant) if relevant else 0.0)

ceiling_500 = np.array(ceiling_500)
print(f"Потолок BM25 top-500 на той же выборке: {ceiling_500.mean():.4f}")
print(f"Наш честный Recall@50:                   {recalls_honest.mean():.4f}")
print(f"Разрыв (потеря на ранжировании):         {ceiling_500.mean() - recalls_honest.mean():.4f}")